In [9]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings("ignore")


In [10]:
print("=" * 55)
print("STAGE 1: FEATURE ENGINEERING PIPELINE")
print("=" * 55)

RAW_PATH = "/content/drive/MyDrive/gridlock_flipkart/jan to may police violation_anonymized791b166.csv"
OUT_PATH  = "/content/drive/MyDrive/gridlock_flipkart/enriched_violations.csv"

print("\n[1/7] Loading raw data...")
df = pd.read_csv(RAW_PATH)
print(f"      Raw records : {len(df):,}")

# Keep only approved records — rejected/NaN challans are not ground truth
df = df[df["validation_status"] == "approved"].copy()
df.reset_index(drop=True, inplace=True)
print(f"      After filter: {len(df):,} approved records")

STAGE 1: FEATURE ENGINEERING PIPELINE

[1/7] Loading raw data...
      Raw records : 298,450
      After filter: 115,400 approved records


In [11]:

print("\n[2/7] Extracting datetime features...")

df["created_dt"] = pd.to_datetime(df["created_datetime"], format="mixed", utc=True)

df["hour"]        = df["created_dt"].dt.hour          # 0–23
df["day_of_week"] = df["created_dt"].dt.dayofweek     # 0=Mon … 6=Sun
df["day_name"]    = df["created_dt"].dt.day_name()
df["month"]       = df["created_dt"].dt.month
df["month_name"]  = df["created_dt"].dt.month_name()
df["date"]        = df["created_dt"].dt.date

# Time-of-day bin — operationally meaningful for patrol scheduling
def time_bin(h):
    if   0 <= h < 6:  return "night"       # peak violation window
    elif 6 <= h < 10: return "morning"
    elif 10 <= h < 16: return "midday"
    elif 16 <= h < 21: return "evening"
    else:              return "late_night"

df["time_bin"] = df["hour"].apply(time_bin)

# Is weekend flag
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

print(f"      Hour range  : {df['hour'].min()}–{df['hour'].max()}")
print(f"      Date range  : {df['date'].min()} to {df['date'].max()}")
print(f"      Night violations (0–5am): "
      f"{(df['time_bin']=='night').sum():,} "
      f"({(df['time_bin']=='night').mean()*100:.1f}%)")


[2/7] Extracting datetime features...
      Hour range  : 0–23
      Date range  : 2023-11-09 to 2024-03-29
      Night violations (0–5am): 61,386 (53.2%)


In [12]:

print("\n[3/7] Parsing violation types...")

# Severity map — based on congestion impact, not fine amount
SEVERITY_MAP = {
    "DOUBLE PARKING":                               4,   # blocks entire lane
    "PARKING IN A MAIN ROAD":                       4,   # arterial blockage
    "PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS":   3,   # intersection risk
    "PARKING NEAR ROAD CROSSING":                  3,
    "PARKING OPPOSITE TO ANOTHER PARKED VEHICLE":  3,   # narrows to 1 lane
    "NO PARKING":                                   2,
    "WRONG PARKING":                                2,
    "PARKING ON FOOTPATH":                          2,   # pushes peds to road
    "PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC":     2,
    "PARKING OTHER THAN BUS STOP":                 1,
    "DEFECTIVE NUMBER PLATE":                       0,   # not a parking violation
    "REFUSE TO GO FOR HIRE":                        0,
    "DEMANDING EXCESS FARE":                        0,
    "USING BLACK FILM/OTHER MATERIALS":             0,
    "WITHOUT SIDE MIRROR":                          0,
    "H T V PROHIBITED":                             1,
    "OBSTRUCTING DRIVER":                           2,
    "AGAINST ONE WAY/NO ENTRY":                     2,
    "FAIL TO USE SAFETY BELTS":                     0,
    "VIOLATING LANE DISIPLINE":                     1,
}

def parse_violations(raw):
    """Parse JSON array string → Python list of violation strings."""
    try:
        return ast.literal_eval(raw)
    except Exception:
        return []

def max_severity(violations):
    """Return highest severity score across all violations in one challan."""
    scores = [SEVERITY_MAP.get(v, 1) for v in violations]
    return max(scores) if scores else 1

def has_parking_violation(violations):
    """Filter: does this challan contain at least one parking-related offence?"""
    parking_keywords = ["PARKING", "DOUBLE", "OBSTRUCTING"]
    return any(
        any(kw in v for kw in parking_keywords)
        for v in violations
    )

df["violations_list"]    = df["violation_type"].apply(parse_violations)
df["violation_count"]    = df["violations_list"].apply(len)
df["severity_score"]     = df["violations_list"].apply(max_severity)
df["is_parking_related"] = df["violations_list"].apply(has_parking_violation).astype(int)

# Keep parking-related only — non-parking challans (helmet, seat belt)
# are enforcement noise for this problem
df = df[df["is_parking_related"] == 1].copy()
df.reset_index(drop=True, inplace=True)

print(f"      After parking filter: {len(df):,} records")
print(f"      Severity distribution:")
print(df["severity_score"].value_counts().sort_index()
      .to_string(header=False))



[3/7] Parsing violation types...
      After parking filter: 115,400 records
      Severity distribution:
2    105990
3       478
4      8932


In [13]:
print("\n[4/7] Assigning vehicle congestion weights...")

# Weight = road-width blocking potential
# Source: IRC road width standards — car occupies ~2.5m, scooter ~0.8m
VEHICLE_WEIGHT = {
    "CAR":                  1.0,
    "JEEP":                 1.0,
    "VAN":                  1.0,
    "MAXI-CAB":             1.2,
    "PASSENGER AUTO":       0.7,
    "GOODS AUTO":           0.7,
    "SCOOTER":              0.4,
    "MOTOR CYCLE":          0.4,
    "MOPED":                0.3,
    "LGV":                  1.5,   # light goods vehicle
    "HGV":                  2.0,   # heavy goods vehicle
    "LORRY/GOODS VEHICLE":  2.0,
    "MINI LORRY":           1.8,
    "TEMPO":                1.3,
    "PRIVATE BUS":          2.5,
    "BUS (BMTC/KSRTC)":     2.5,
    "TOURIST BUS":          2.5,
    "SCHOOL VEHICLE":       2.0,
    "FACTORY BUS":          2.5,
    "TANKER":               2.0,
    "TRACTOR":              1.5,
    "OTHERS":               1.0,
}

df["vehicle_weight"] = df["vehicle_type"].map(VEHICLE_WEIGHT).fillna(1.0)

print("      Vehicle weight summary:")
wt = df.groupby("vehicle_type")["vehicle_weight"].first().sort_values(ascending=False)
for vt, w in wt.items():
    count = (df["vehicle_type"] == vt).sum()
    print(f"        {vt:<25} weight={w:.1f}  count={count:,}")



[4/7] Assigning vehicle congestion weights...
      Vehicle weight summary:
        BUS (BMTC/KSRTC)          weight=2.5  count=287
        FACTORY BUS               weight=2.5  count=76
        PRIVATE BUS               weight=2.5  count=568
        TOURIST BUS               weight=2.5  count=105
        HGV                       weight=2.0  count=413
        LORRY/GOODS VEHICLE       weight=2.0  count=446
        SCHOOL VEHICLE            weight=2.0  count=148
        TANKER                    weight=2.0  count=103
        MINI LORRY                weight=1.8  count=72
        LGV                       weight=1.5  count=3,153
        TRACTOR                   weight=1.5  count=21
        TEMPO                     weight=1.3  count=461
        MAXI-CAB                  weight=1.2  count=4,877
        CAR                       weight=1.0  count=36,833
        VAN                       weight=1.0  count=546
        JEEP                      weight=1.0  count=377
        OTHERS         

In [14]:

print("\n[5/7] Computing junction scores...")

# Binary: at a named junction = 1, No Junction = 0
# Named junctions come with BTP codes (e.g. BTP044 - Sagar Theatre Junction)
df["at_junction"] = (
    df["junction_name"].fillna("No Junction") != "No Junction"
).astype(int)

# Extract junction code for later grouping (e.g. "BTP044")
def extract_junction_code(name):
    if pd.isna(name) or name == "No Junction":
        return None
    parts = str(name).split(" - ")
    return parts[0].strip() if len(parts) > 1 else None

df["junction_code"] = df["junction_name"].apply(extract_junction_code)

print(f"      At junction    : {df['at_junction'].sum():,} "
      f"({df['at_junction'].mean()*100:.1f}%)")
print(f"      Not at junction: {(df['at_junction']==0).sum():,} "
      f"({(df['at_junction']==0).mean()*100:.1f}%)")
print(f"      Unique junctions: {df['junction_code'].nunique()}")



[5/7] Computing junction scores...
      At junction    : 60,907 (52.8%)
      Not at junction: 54,493 (47.2%)
      Unique junctions: 167


In [15]:
print("\n[6/7] Geo bounds validation...")

# Bengaluru bounding box (generous)
LAT_MIN, LAT_MAX = 12.70, 13.35
LON_MIN, LON_MAX = 77.35, 77.85

outliers = df[
    (df["latitude"]  < LAT_MIN) | (df["latitude"]  > LAT_MAX) |
    (df["longitude"] < LON_MIN) | (df["longitude"] > LON_MAX)
]
print(f"      Geo outliers outside Bengaluru bbox: {len(outliers)}")

if len(outliers) > 0:
    df = df[
        (df["latitude"].between(LAT_MIN, LAT_MAX)) &
        (df["longitude"].between(LON_MIN, LON_MAX))
    ].copy()
    print(f"      Removed. Remaining: {len(df):,}")


[6/7] Geo bounds validation...
      Geo outliers outside Bengaluru bbox: 0


In [16]:
print("\n[7/7] Building final enriched dataset...")

KEEP_COLS = [
    # Identity
    "id",
    # Spatial
    "latitude", "longitude",
    "police_station", "junction_name", "junction_code",
    "at_junction",
    # Temporal
    "created_dt", "date", "hour", "day_of_week", "day_name",
    "month", "month_name", "time_bin", "is_weekend",
    # Violation
    "violation_type", "violations_list", "violation_count",
    "severity_score", "is_parking_related",
    # Vehicle
    "vehicle_type", "vehicle_weight",
    # Meta
    "validation_status",
]

enriched = df[KEEP_COLS].copy()
enriched.to_csv(OUT_PATH, index=False)

print(f"\n{'='*55}")
print(f"ENRICHED DATASET SAVED → {OUT_PATH}")
print(f"{'='*55}")
print(f"  Records          : {len(enriched):,}")
print(f"  Columns          : {len(enriched.columns)}")
print(f"  Date range       : {enriched['date'].min()} to {enriched['date'].max()}")
print(f"\n  Feature columns ready for Stage 2 (HDBSCAN):")
print(f"    Spatial   → latitude, longitude")
print(f"    Temporal  → hour, day_of_week, time_bin")
print(f"    Severity  → severity_score, vehicle_weight")
print(f"    Junction  → at_junction, junction_code")
print(f"\n  Next step → run ps1_hdbscan_clustering.py")
print(f"{'='*55}\n")

# Quick sanity print
print("SAMPLE OUTPUT (5 rows, key columns):")
print(enriched[["id","latitude","longitude","hour","time_bin",
                "severity_score","vehicle_weight","at_junction"]].head())



[7/7] Building final enriched dataset...

ENRICHED DATASET SAVED → /content/drive/MyDrive/gridlock_flipkart/enriched_violations.csv
  Records          : 115,400
  Columns          : 24
  Date range       : 2023-11-09 to 2024-03-29

  Feature columns ready for Stage 2 (HDBSCAN):
    Spatial   → latitude, longitude
    Temporal  → hour, day_of_week, time_bin
    Severity  → severity_score, vehicle_weight
    Junction  → at_junction, junction_code

  Next step → run ps1_hdbscan_clustering.py

SAMPLE OUTPUT (5 rows, key columns):
           id   latitude  longitude  hour time_bin  severity_score  \
0  FKID000000  12.925557  77.618665     0    night               3   
1  FKID000002  12.925449  77.618504     0    night               4   
2  FKID000003  12.956521  77.518618     6  morning               2   
3  FKID000004  12.977767  77.580545     4    night               2   
4  FKID000005  12.981677  77.608125     7  morning               2   

   vehicle_weight  at_junction  
0            